# SatQuery AI — the real Qwen on a free T4

The laptop cannot run the served VLM: Qwen2.5-VL-7B is five safetensors shards,
~16 GB in fp16, and 4-bit needs `bitsandbytes`, which needs CUDA. This notebook
runs it where it fits and exposes the **whole API** — validator, tool menu,
router, executor, composer, GUI — on a public URL, so the demo runs in your
browser against real Qwen answers.

**Before running:** Runtime → Change runtime type → **T4 GPU**. On Kaggle pick
the **2×T4** accelerator, never P100 (Unsloth needs CUDA ≥ 7.0).

Roughly 10 minutes: ~3 to install, ~5 to pull weights, seconds per query after.

What each slot will be running when this is up:

| slot | here | why |
|---|---|---|
| text (5 tasks) | **Qwen2.5-VL-7B, 4-bit** | the point of this notebook |
| router | Mistral, or offline rules | set `MISTRAL_API_KEY` below to use Mistral |
| change map | `cva-otsu` | MCI is not vendored yet — see the last section |
| fusion | `fusion-rules-v1` | TerraMind needs terratorch — see the last section |


## 1 · Confirm the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4'
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f'{name}  sm_{major}{minor}  {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB')
# SM75 is the T4. fp16 + sdpa only: no bf16, no FlashAttention-2 (see config.py).
if (major, minor) < (7, 5):
    print('WARNING: below SM75. 4-bit loading may fail.')


## 2 · Install

The `vlm` extra is torch + transformers + accelerate + bitsandbytes. Colab
already has torch, so this mostly pulls the quantisation stack.


In [ ]:
REPO = 'https://github.com/geek007git/SIH26171.git'
BRANCH = 'main'

import os, pathlib
if not pathlib.Path('SIH26171').exists():
    !git clone --depth 1 -b {BRANCH} {REPO}
%cd SIH26171
!pip -q install -e '.[data,api,vlm]' 2>&1 | tail -3
!pip -q install qwen-vl-utils 2>&1 | tail -1


## 3 · Does the package see the GPU?

`doctor` reports which backend would serve each slot and why. With CUDA visible,
the text slot should flip from `evidence-rules` to Qwen on its own.


In [ ]:
import os
os.environ['SATQUERY_VLM'] = 'qwen2_5_vl'   # explicit rather than relying on auto
# Optional: routing via Mistral instead of the offline keyword router.
# os.environ['MISTRAL_API_KEY'] = 'paste-here-or-leave-unset'

!python -m satquery.cli doctor


## 4 · Demo imagery

`demo/` is gitignored — the generator is deterministic, so one command rebuilds
the exact scenes used elsewhere.


In [ ]:
!python tools/make_demo_scenes.py

from IPython.display import Image, display
display(Image('demo/city_2019.png', width=320), Image('demo/city_2021.png', width=320))


## 5 · Load Qwen once, and time it

The predictor loads lazily on first use and stays resident. Expect ~7 GB of the
T4's 15 GB at 4-bit, and a minute or two for the first call including the
download.


In [ ]:
import time
from satquery.registry import build_predictor
from satquery.contracts import ImageRef, Sample

predictor = build_predictor('qwen2_5_vl')   # Qwen/Qwen2.5-VL-7B-Instruct, 4-bit
print(predictor.describe())

sample = Sample(
    sample_id='demo',
    task='caption',
    images=(ImageRef(image_id='a', path='demo/city_2019.png', modality='optical'),),
)
t0 = time.perf_counter()
prediction = predictor.predict(sample)
print(f'first call (includes load): {time.perf_counter()-t0:.1f}s')
print('caption:', prediction.text)
print('confidence:', prediction.confidence)
print('VRAM:', f'{torch.cuda.max_memory_allocated()/2**30:.1f} GB peak')


## 6 · The whole loop, with Qwen behind the text slot

Same controller as on the laptop; only the backend differs. Compare the answer
and the citations with what `evidence-rules` produced for the same query.


In [ ]:
from satquery.agent.session import SessionStore
from satquery.agent.controller import Controller

session = SessionStore().get('gpu')
a = session.add_image('demo/city_2019.png')
b = session.add_image('demo/city_2021.png')

for question in ['What changed between these two dates?',
                 'How many buildings are in the town?',
                 'Where is the reservoir?']:
    result = Controller(session).answer(question, image_ids=[a.image_id, b.image_id])
    print('Q:', question)
    print('  backend :', result.backends['text']['backend'])
    print('  plan    :', result.plan['tools'])
    print('  answer  :', result.answer[:300])
    print('  cited   :', [(c['claim'], c['key']) for c in result.citations][:4])
    print('  conf    :', result.confidence)
    print()


## 7 · Serve the GUI on a public URL

A Cloudflare quick tunnel — no account, no signup. The printed
`https://<something>.trycloudflare.com` is the whole app: upload, ask, trace,
PDF. Open it on the laptop and demo against this GPU.

Keep this cell running. Stop it to close the tunnel.


In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

import os, subprocess, sys, time

server = subprocess.Popen(
    [sys.executable, '-m', 'satquery.cli', 'serve', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ},
)
time.sleep(8)

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for line in tunnel.stdout:
    # A substring scan rather than a regex: a backslash in a notebook source
    # line survives two layers of JSON escaping badly, and this is clearer.
    if 'trycloudflare.com' in line and 'https://' in line:
        url = ('https://' + line.split('https://', 1)[1].split()[0]).rstrip('|,. ')
        print()
        print('  OPEN THIS:', url)
        print()
        break
    if 'ERR' in line:
        print(line.rstrip())


## 8 · Optional — the before/after benchmark number

The claim that gets scored is the *delta* on a prescribed split, measured by the
same harness both times. Point a loader at a download first (`satquery datasets`
shows what is missing), then:

```python
!python -m satquery.cli eval --dataset rsvqa_lr --predictor qwen2_5_vl --limit 200
!python -m satquery.cli compare runs/<baseline> runs/<tuned>
```

Read **both** accuracy columns. An untuned model answers in sentences and a
tuned one answers tersely, so a strict-only comparison credits the fine-tune for
learning the output format — `compare` prints strict and lenient side by side
and flags when they disagree in direction.


## 9 · The other two slots

Neither is wired yet, and both are honest about it: selecting them explicitly
raises with a reason rather than an `ImportError`.

**Change map — Change-Agent MCI.** Hugging Face ships `MCI_model.pth` and
nothing else; the model definition lives in the GitHub repo on an mmseg/mmcv
stack that is version-pinned to older torch. Budget a session for the install
alone. Until then `cva-otsu` measures the mask deterministically — on synthetic
scenes it reports 7.32% for a 7.32% change and 0.0% for an unchanged pair.

**Fusion — TerraMind.** Better than the plan assumed. `CONTEXT.md` budgeted a
training run on BigEarthNet-MM for a task head, but TerraMind is any-to-any and
there is a published `TerraMind-1.0-Tokenizer-LULC`, so generating land cover
from S2L2A + S1GRD may need no head at all. Worth an hour before committing to
the training run:

```python
!pip -q install terratorch    # 68 dependencies, allow time
# then: terratorch.FULL_MODEL_REGISTRY -> TerraMind generation, S1GRD+S2L2A -> LULC
```

Both slots take a `detect()` / `classify()` implementation and nothing else —
`agent/backends.py` already defines the classes, the selection policy and the
result types, and `agent/tools.py` already calls them.
